In [1]:
import sys
sys.path.append('../src/')

In [2]:
import pystac_client
import planetary_computer
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

from rasterio.enums import Resampling
import stackstac
from pathlib import Path
from pystac.extensions.eo import EOExtension as eo
from utils import intersection_percent, num_day_to_datetime

In [45]:
# mtbs = gpd.read_file("/mnt/sherlock/prescribed_data/geoms/mtbs_perims_DD/")
# mtbs["Ig_Date"] = pd.to_datetime(mtbs["Ig_Date"])
# mtbs_ca = mtbs[(mtbs.Event_ID.str.contains("CA")) & (mtbs.Incid_Type == "Wildfire")]

# mtbs_ca_meters = mtbs_ca.to_crs(3310)

# mtbs_ca_buffers = mtbs_ca.copy()
# mtbs_ca_buffers["geometry"] = (
#     mtbs_ca_meters
#     .buffer(180)
#     .to_crs(mtbs_ca.crs)
# )

dnbr_files = list(Path("/mnt/sherlock/prescribed_data/processed/dnbr_landsat_c2_l2_buf/").rglob("*.tif"))
#dnbr_files = list(Path("../data/dnbr_offset/").rglob("*.tif"))
dnbr_events = [d.name.split(".")[0] for d in dnbr_files]

In [46]:
not_observed = mtbs_ca[~mtbs_ca.Event_ID.isin(dnbr_events)]
not_observed.sort_values("BurnBndAc", ascending=False, inplace=True)

/tmp/ipykernel_803344/2024885120.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  not_observed.sort_values("BurnBndAc", ascending=False, inplace=True)


In [47]:
not_observed.shape

(107, 23)

In [44]:
not_observed.groupby(not_observed.Ig_Date.dt.year).Event_ID.count()

Ig_Date
1984    42
1985    12
1989     1
2000     1
2007     1
2011    22
2013    28
Name: Event_ID, dtype: int64

In [42]:
not_observed[["Event_ID", "Incid_Name", "Ig_Date"]].head(20)

,Event_ID,Incid_Name,Ig_Date
21780,CA3785712008620130817,RIM,2013-08-17
578,CA3451011935419850701,WHEELER 2,1985-07-01
21558,CA3459511843720130530,POWERHOUSE,2013-05-30
21681,CA3370511672620130715,MOUNTAIN,2013-07-15
21760,CA3911912064620130810,AMERICAN,2013-08-10
20144,CA3539511864020110910,BRECKENRIDGE COMPLEX,2011-09-10
20142,CA3511711878320110910,COMANCHE,2011-09-10
21730,CA4133312337620130731,BUTLER,2013-07-31
21503,CA3420011896420130502,SPRINGS,2013-05-02
21703,CA3727911931820130722,ASPEN,2013-07-22


In [29]:
start_time_str = (
    num_day_to_datetime(152, not_observed.iloc[0].Ig_Date.year - 1)
    .strftime("%Y-%m-%d")
)
end_time_str = (
    num_day_to_datetime(273, not_observed.iloc[0].Ig_Date.year - 1)
    .strftime("%Y-%m-%d")
)

timerange = f"{start_time_str}/{end_time_str}"
bounds = not_observed.iloc[0].geometry.bounds

In [30]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

In [31]:


search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=not_observed.iloc[0].geometry.bounds,
    datetime=timerange,
    query={"eo:cloud_cover": {"lt": 40},
           "platform": {"in": ["landsat-4", "landsat-5", "landsat-8", "landsat-9"]}
          },
)

items_search = search.get_all_items()

In [32]:
item_aoi_coverage = [
    intersection_percent(item, not_observed.iloc[0].geometry) for item in items_search.items
]

In [33]:
item_aoi_coverage

[]

In [28]:
filter_items = [item for 
              item, coverage 
              in zip(items_search, item_aoi_coverage)
              if float(coverage) > 99
             ]
filter_items

[<Item id=LC08_L2SP_043034_20140919_02_T1>,
 <Item id=LC08_L2SP_043034_20140903_02_T1>,
 <Item id=LC08_L2SP_043034_20140818_02_T1>,
 <Item id=LC08_L2SP_043034_20140802_02_T1>,
 <Item id=LC08_L2SP_043034_20140717_02_T1>,
 <Item id=LC08_L2SP_043034_20140701_02_T1>,
 <Item id=LC08_L2SP_043034_20140615_02_T1>]

In [17]:
from dask.distributed import Client

client = Client("tcp://127.0.0.1:44933")
client

<Client: 'tcp://127.0.0.1:44933' processes=5 threads=20, memory=62.44 GiB>

In [ ]:
data = stackstac.stack(
    filter_items,
    bounds=not_observed.iloc[0]["geometry"].bounds,
    assets= ["blue", "green", "red", "nir08", "swir16", "qa_pixel"],
    epsg=4326,
    resampling=Resampling.bilinear,
    snap_bounds=True

)
#data = data.compute()

In [ ]:
data.drop("proj:shape")

In [ ]:
data.attrs = {}
data = data.drop_vars(
    [
        "inst"
    ], errors="ignore"
)


In [ ]:
data.to_netcdf("test.nc")

In [ ]:
(
    data.mean(dim="time")
    .sel(band=["red", "green", "blue"])
    .plot.imshow(robust=True)
)

